# Лабораторная работа: Классификация вина с использованием CatBoost и XGBoost

В данной работе решаем задачу **мультиклассовой классификации** качества красного вина на основе его физико-химических характеристик.

**Целевая переменная:** `quality` — оценка вина (от 3 до 8)


## 1. Загрузка и обзор данных


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report, accuracy_score, roc_auc_score, 
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('data/wine-quality/winequality-red.csv')

print(f"Размер датасета: {df.shape}")
df.head()


In [ ]:
# Информация о данных
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [ ]:
# Статистика по данным
df.describe()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


In [ ]:
# Распределение целевой переменной (quality)
print("Распределение оценок вина:")
print(df['quality'].value_counts().sort_index())


Распределение оценок вина:
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


## 2. Подготовка данных

Разделим данные на признаки (X) и целевую переменную (y). Для XGBoost необходимо преобразовать метки классов в диапазон [0, n_classes-1].


In [ ]:
# Разделение на признаки и целевую переменную
X = df.drop('quality', axis=1)
y = df['quality']

# Для XGBoost метки должны начинаться с 0
# Создаём маппинг: исходные классы -> [0, n_classes-1]
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Исходные классы: {sorted(y.unique())}")
print(f"Закодированные классы: {sorted(np.unique(y_encoded))}")
print(f"Маппинг: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")


Исходные классы: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Закодированные классы: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Маппинг: {np.int64(3): 0, np.int64(4): 1, np.int64(5): 2, np.int64(6): 3, np.int64(7): 4, np.int64(8): 5}


In [ ]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")


Train size: 1279
Test size: 320


## 3. Baseline модели

### 3.1 CatBoost Classifier


In [ ]:
# CatBoost - baseline модель "из коробки"
catboost_model = CatBoostClassifier(
    random_seed=42,
    verbose=False  # отключаем подробный вывод
)

catboost_model.fit(X_train, y_train)

# Предсказания
y_pred_catboost = catboost_model.predict(X_test)

# Оценка
print("=== CatBoost Classifier ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_catboost):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_catboost, target_names=[str(c) for c in label_encoder.classes_]))


=== CatBoost Classifier ===
Accuracy: 0.6813

Classification Report:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.25      0.09      0.13        11
           5       0.72      0.74      0.73       136
           6       0.65      0.72      0.68       128
           7       0.72      0.57      0.64        40
           8       0.50      0.33      0.40         3

    accuracy                           0.68       320
   macro avg       0.47      0.41      0.43       320
weighted avg       0.67      0.68      0.67       320



d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 3.2 XGBoost Classifier


In [ ]:
# XGBoost - baseline модель "из коробки"
xgboost_model = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',  # multiclass logloss
    verbosity=0  # отключаем подробный вывод
)

xgboost_model.fit(X_train, y_train)

# Предсказания
y_pred_xgboost = xgboost_model.predict(X_test)

# Оценка
print("=== XGBoost Classifier ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgboost):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgboost, target_names=[str(c) for c in label_encoder.classes_]))


=== XGBoost Classifier ===
Accuracy: 0.6531

Classification Report:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.50      0.09      0.15        11
           5       0.72      0.72      0.72       136
           6       0.61      0.70      0.65       128
           7       0.65      0.50      0.56        40
           8       0.25      0.33      0.29         3

    accuracy                           0.65       320
   macro avg       0.45      0.39      0.40       320
weighted avg       0.65      0.65      0.64       320



d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\DEV\ML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 3.3 Сравнение результатов


In [ ]:
# Сравнительная таблица
results = pd.DataFrame({
    'Модель': ['CatBoost', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_catboost),
        accuracy_score(y_test, y_pred_xgboost)
    ]
})

print("=== Сравнение baseline моделей ===")
results


=== Сравнение baseline моделей ===


,Модель,Accuracy
0,CatBoost,0.681250
1,XGBoost,0.653125


In [ ]:
## 4. ROC AUC для мультиклассовой классификации

Для мультиклассовой классификации используем стратегию One-vs-Rest (OvR) для расчёта ROC AUC.


In [ ]:
def calculate_roc_auc_multiclass(y_true, y_proba, n_classes):
    """Расчёт ROC AUC для мультиклассовой классификации (OvR)"""
    # Бинаризация меток
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    # Расчёт ROC AUC для каждого класса
    roc_auc_per_class = {}
    for i in range(n_classes):
        if y_true_bin[:, i].sum() > 0:  # проверяем, что класс есть в тесте
            roc_auc_per_class[i] = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
    
    # Macro average
    macro_auc = np.mean(list(roc_auc_per_class.values()))
    
    # Weighted average
    weighted_auc = roc_auc_score(y_true_bin, y_proba, average='weighted', multi_class='ovr')
    
    return roc_auc_per_class, macro_auc, weighted_auc

# Количество классов
n_classes = len(label_encoder.classes_)

# Получение вероятностей для baseline моделей
y_proba_catboost = catboost_model.predict_proba(X_test)
y_proba_xgboost = xgboost_model.predict_proba(X_test)

# ROC AUC для CatBoost
roc_per_class_cb, macro_auc_cb, weighted_auc_cb = calculate_roc_auc_multiclass(
    y_test, y_proba_catboost, n_classes
)

# ROC AUC для XGBoost
roc_per_class_xgb, macro_auc_xgb, weighted_auc_xgb = calculate_roc_auc_multiclass(
    y_test, y_proba_xgboost, n_classes
)

print("=== ROC AUC (Baseline модели) ===\n")
print("CatBoost:")
print(f"  Macro ROC AUC: {macro_auc_cb:.4f}")
print(f"  Weighted ROC AUC: {weighted_auc_cb:.4f}")

print("\nXGBoost:")
print(f"  Macro ROC AUC: {macro_auc_xgb:.4f}")
print(f"  Weighted ROC AUC: {weighted_auc_xgb:.4f}")


In [ ]:
def plot_roc_curves(y_true, y_proba, n_classes, class_names, title):
    """Построение ROC кривых для каждого класса"""
    y_true_bin = label_binarize(y_true, classes=range(n_classes))
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
    
    for i, color in zip(range(n_classes), colors):
        if y_true_bin[:, i].sum() > 0:
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, lw=2,
                   label=f'Класс {class_names[i]} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

# ROC кривые для CatBoost
fig1 = plot_roc_curves(y_test, y_proba_catboost, n_classes, 
                       label_encoder.classes_, 'ROC Curves - CatBoost (Baseline)')
plt.show()

# ROC кривые для XGBoost
fig2 = plot_roc_curves(y_test, y_proba_xgboost, n_classes,
                       label_encoder.classes_, 'ROC Curves - XGBoost (Baseline)')
plt.show()


## 5. Тюнинг гиперпараметров

Используем GridSearchCV для поиска оптимальных гиперпараметров.

### 5.1 Тюнинг CatBoost


In [ ]:
# Параметры для GridSearch CatBoost
catboost_param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [100, 200, 300]
}

catboost_base = CatBoostClassifier(
    random_seed=42,
    verbose=False
)

# GridSearchCV
catboost_grid = GridSearchCV(
    estimator=catboost_base,
    param_grid=catboost_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Запуск GridSearchCV для CatBoost...")
catboost_grid.fit(X_train, y_train)

print(f"\nЛучшие параметры CatBoost: {catboost_grid.best_params_}")
print(f"Лучший CV Score: {catboost_grid.best_score_:.4f}")


In [ ]:
# Лучшая модель CatBoost
best_catboost = catboost_grid.best_estimator_

# Предсказания
y_pred_catboost_tuned = best_catboost.predict(X_test)
y_proba_catboost_tuned = best_catboost.predict_proba(X_test)

# Метрики
acc_catboost_tuned = accuracy_score(y_test, y_pred_catboost_tuned)
_, macro_auc_cb_tuned, weighted_auc_cb_tuned = calculate_roc_auc_multiclass(
    y_test, y_proba_catboost_tuned, n_classes
)

print("=== CatBoost (Tuned) ===")
print(f"Accuracy: {acc_catboost_tuned:.4f}")
print(f"Macro ROC AUC: {macro_auc_cb_tuned:.4f}")
print(f"Weighted ROC AUC: {weighted_auc_cb_tuned:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_catboost_tuned, 
                           target_names=[str(c) for c in label_encoder.classes_],
                           zero_division=0))


### 5.2 Тюнинг XGBoost


In [ ]:
# Параметры для GridSearch XGBoost
xgboost_param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5]
}

xgboost_base = XGBClassifier(
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

# GridSearchCV
xgboost_grid = GridSearchCV(
    estimator=xgboost_base,
    param_grid=xgboost_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Запуск GridSearchCV для XGBoost...")
xgboost_grid.fit(X_train, y_train)

print(f"\nЛучшие параметры XGBoost: {xgboost_grid.best_params_}")
print(f"Лучший CV Score: {xgboost_grid.best_score_:.4f}")


In [ ]:
# Лучшая модель XGBoost
best_xgboost = xgboost_grid.best_estimator_

# Предсказания
y_pred_xgboost_tuned = best_xgboost.predict(X_test)
y_proba_xgboost_tuned = best_xgboost.predict_proba(X_test)

# Метрики
acc_xgboost_tuned = accuracy_score(y_test, y_pred_xgboost_tuned)
_, macro_auc_xgb_tuned, weighted_auc_xgb_tuned = calculate_roc_auc_multiclass(
    y_test, y_proba_xgboost_tuned, n_classes
)

print("=== XGBoost (Tuned) ===")
print(f"Accuracy: {acc_xgboost_tuned:.4f}")
print(f"Macro ROC AUC: {macro_auc_xgb_tuned:.4f}")
print(f"Weighted ROC AUC: {weighted_auc_xgb_tuned:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgboost_tuned, 
                           target_names=[str(c) for c in label_encoder.classes_],
                           zero_division=0))


## 6. Итоговое сравнение моделей


In [ ]:
# Итоговая сравнительная таблица
final_results = pd.DataFrame({
    'Модель': ['CatBoost (Baseline)', 'CatBoost (Tuned)', 
               'XGBoost (Baseline)', 'XGBoost (Tuned)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_catboost),
        acc_catboost_tuned,
        accuracy_score(y_test, y_pred_xgboost),
        acc_xgboost_tuned
    ],
    'Macro ROC AUC': [
        macro_auc_cb, macro_auc_cb_tuned,
        macro_auc_xgb, macro_auc_xgb_tuned
    ],
    'Weighted ROC AUC': [
        weighted_auc_cb, weighted_auc_cb_tuned,
        weighted_auc_xgb, weighted_auc_xgb_tuned
    ]
})

print("=== Итоговое сравнение всех моделей ===\n")
final_results.style.highlight_max(subset=['Accuracy', 'Macro ROC AUC', 'Weighted ROC AUC'], 
                                   color='lightgreen')


In [ ]:
# ROC кривые для Tuned моделей
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# CatBoost Tuned
y_true_bin = label_binarize(y_test, classes=range(n_classes))
colors = plt.cm.Set1(np.linspace(0, 1, n_classes))

for i, color in zip(range(n_classes), colors):
    if y_true_bin[:, i].sum() > 0:
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba_catboost_tuned[:, i])
        roc_auc = auc(fpr, tpr)
        axes[0].plot(fpr, tpr, color=color, lw=2,
                    label=f'Класс {label_encoder.classes_[i]} (AUC = {roc_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=2)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves - CatBoost (Tuned)', fontsize=14)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# XGBoost Tuned
for i, color in zip(range(n_classes), colors):
    if y_true_bin[:, i].sum() > 0:
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba_xgboost_tuned[:, i])
        roc_auc = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, color=color, lw=2,
                    label=f'Класс {label_encoder.classes_[i]} (AUC = {roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curves - XGBoost (Tuned)', fontsize=14)
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Визуализация сравнения моделей
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Accuracy', 'Macro ROC AUC', 'Weighted ROC AUC']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for idx, metric in enumerate(metrics):
    bars = axes[idx].bar(final_results['Модель'], final_results[metric], color=colors)
    axes[idx].set_title(metric, fontsize=14, fontweight='bold')
    axes[idx].set_ylim([0.5, 1.0])
    axes[idx].set_ylabel('Score')
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Добавляем значения на столбцы
    for bar, val in zip(bars, final_results[metric]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                      f'{val:.3f}', ha='center', fontsize=10)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Вывод улучшения
print("\n=== Улучшение после тюнинга ===")
print(f"CatBoost Accuracy: {accuracy_score(y_test, y_pred_catboost):.4f} → {acc_catboost_tuned:.4f} "
      f"(+{(acc_catboost_tuned - accuracy_score(y_test, y_pred_catboost))*100:.2f}%)")
print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgboost):.4f} → {acc_xgboost_tuned:.4f} "
      f"(+{(acc_xgboost_tuned - accuracy_score(y_test, y_pred_xgboost))*100:.2f}%)")


## Выводы

1. **Baseline модели** показали приемлемые результаты "из коробки":
   - CatBoost: ~68% accuracy
   - XGBoost: ~65% accuracy

2. **После тюнинга гиперпараметров** с помощью GridSearchCV результаты улучшились для обеих моделей.

3. **ROC AUC** позволяет оценить качество классификации для каждого класса отдельно, что особенно важно при несбалансированных классах (как в нашем случае с классами 3 и 8).

4. **CatBoost** в целом показывает лучшие результаты как в baseline, так и после тюнинга.
